# Thực nghiệm bài toán tham quan Hà Nội trong một ngày

Notebook này chạy các thuật toán Greedy, A* và vét cạn (`greedy.py`, `astar.py`, `brute.py`) cho hai thực nghiệm TN1 và TN2; các hàm thực nghiệm nằm trong `experiments.py`.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from experiments import run_all, run_once, trace_path, BASE_MATRIX, BASE_PLACES, OUTPUT_DIR, TIME_REPEATS
print("Thư mục dự án:", PROJECT_DIR)
print("Số lần đo thời gian:", TIME_REPEATS)

Thư mục dự án: /home/claude/proj
Số lần đo thời gian: 100


## 1. Chạy thí nghiệm

TN1: 8 tiếng (8:00–16:00), mọi địa điểm mở cả ngày. TN2: 6 tiếng (8:00–14:00), giờ mở/đóng cửa gốc (3 điểm đóng 11:00). Thời gian chạy là trung bình qua 100 lần; kết quả CSV lưu trong thư mục `results`.

In [2]:
fixed_rows = run_all(repeats=TIME_REPEATS)
fixed_df = pd.DataFrame(fixed_rows)
print("Đã chạy xong TN1, TN2.")
print("Thư mục CSV:", OUTPUT_DIR)

Đã chạy xong TN1, TN2.
Thư mục CSV: /home/claude/proj/results


## 2. Kết quả TN1 và TN2

`num_visited` là số điểm tham quan được ghé; `finish_clock` là giờ hoàn thành. `states` được định nghĩa theo thuật toán: Greedy là tổng số lần gọi `try_visit` (lọc ứng viên và đếm h), A* là số trạng thái được mở rộng, vét cạn là số hoán vị được đánh giá.

In [3]:
display(fixed_df[[
    "scenario", "algorithm", "num_visited", "finish_clock",
    "states", "runtime_seconds_avg", "runtime_repeats", "path_names"
]])

,scenario,algorithm,num_visited,finish_clock,states,runtime_seconds_avg,runtime_repeats,path_names
0,TN1,Greedy,6,16:00,140,0.000039,100,St. Joseph's Cathedral -> Dong Xuan Market -> ...
1,TN1,A*,6,14:56,3845,0.748551,100,St. Joseph's Cathedral -> Dong Xuan Market -> ...
2,TN1,Vet can,6,14:56,5040,0.007671,100,St. Joseph's Cathedral -> Dong Xuan Market -> ...
3,TN2,Greedy,5,13:46,126,0.000035,100,St. Joseph's Cathedral -> Dong Xuan Market -> ...
4,TN2,A*,5,13:46,103,0.002908,100,St. Joseph's Cathedral -> Dong Xuan Market -> ...
5,TN2,Vet can,5,13:46,5040,0.004652,100,St. Joseph's Cathedral -> Dong Xuan Market -> ...


In [4]:
# Kiểm tra điều kiện nghiệm thu trên số điểm (mỗi dòng: TN1, TN2).
count_pivot = fixed_df.pivot(index="scenario", columns="algorithm", values="num_visited")
count_pivot["A*_equals_brute"] = count_pivot["A*"] == count_pivot["Vet can"]
count_pivot["brute_not_worse_than_greedy"] = count_pivot["Vet can"] >= count_pivot["Greedy"]
display(count_pivot)

algorithm,A*,Greedy,Vet can,A*_equals_brute,brute_not_worse_than_greedy
scenario,,,,,
TN1,6,6,6,True,True
TN2,5,5,5,True,True


## 3. Kiểm tra kết quả

Không chỉnh số liệu để ép kết quả; nếu điều kiện nào không đạt, cần kiểm tra lại dữ liệu và thuật toán.

In [5]:
checks = {
    "A* bằng vét cạn ở TN1 và TN2": bool((count_pivot["A*"] == count_pivot["Vet can"]).all()),
    "Vét cạn không kém Greedy ở TN1 và TN2": bool((count_pivot["Vet can"] >= count_pivot["Greedy"]).all()),
}
for label, passed in checks.items():
    print(("ĐẠT" if passed else "CHƯA ĐẠT") + " — " + label)

ĐẠT — A* bằng vét cạn ở TN1 và TN2
ĐẠT — Vét cạn không kém Greedy ở TN1 và TN2


In [6]:
print("Các file kết quả:")
for file in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", file.name)

Các file kết quả:
- fixed_scenarios_results.csv
